# GeoLab Tutorial: Converting Decimal Degrees to DMS (Degrees, Minutes, Seconds)

This notebook converts geographic coordinates stored as **decimal degrees** (e.g., `103.411599`) into the classic **DMS format** (e.g., `103°24'41.76"`) that is commonly used in field reports, GPS devices, and Indonesian government GIS forms.

**Output:** A new CSV file containing all original columns plus two new columns — `latitude_dms` and `longitude_dms` — with coordinates formatted as human-readable DMS strings.

> **Target audience:** GIS/QGIS users who are beginning to learn geospatial programming in Python.

## Requirements

Install the required packages before running this notebook:

```bash
pip install pandas
```

Or using conda:

```bash
conda install -c conda-forge pandas
```

| Package | Purpose |
|---|---|
| `pandas` | Reading the input CSV and writing the output CSV with new DMS columns |

## Step 1: Import Libraries

In [ ]:
# pandas is the standard Python library for working with tabular data (like a spreadsheet).
# We use it to load the CSV, add the new DMS columns, and save the result.
import pandas as pd

## Step 2: Define the Conversion Function

The conversion from Decimal Degrees (DD) to Degrees-Minutes-Seconds (DMS) is a standard geographic formula:

- **Degrees** = the integer part of the decimal value  
- **Minutes** = the integer part of the remaining fraction × 60  
- **Seconds** = the remaining fraction after removing degrees and minutes × 3600

For example: `103.6922° → 103° 41' 31.92"`

In [ ]:
def decimal_to_dms(decimal_degrees: float) -> str:
    """
    Convert a geographic coordinate from Decimal Degrees to DMS format.

    This function handles only the magnitude of the coordinate (unsigned).
    For signed values (negative latitudes/longitudes), the sign should be
    handled separately before calling this function if needed.

    Parameters
    ----------
    decimal_degrees : float
        A coordinate value in decimal degrees (e.g., 103.6922 or -0.6691).

    Returns
    -------
    str
        The coordinate in DMS string format, e.g. "103°41'31.92\""

    Examples
    --------
    >>> decimal_to_dms(103.6922)
    "103°41'31.92\""
    >>> decimal_to_dms(-0.6691)
    "-0°40'8.76\""
    """
    # Preserve the sign to handle negative values (southern latitudes / western longitudes)
    sign = -1 if decimal_degrees < 0 else 1
    decimal_degrees = abs(decimal_degrees)

    # Degrees: integer part of the decimal value
    degrees = int(decimal_degrees)

    # Minutes: take the leftover fraction, multiply by 60, take the integer part
    minutes_float = (decimal_degrees - degrees) * 60
    minutes = int(minutes_float)

    # Seconds: take the leftover fraction after minutes, multiply by 60
    seconds = (minutes_float - minutes) * 60

    # Re-apply the sign to the degrees component
    degrees = sign * degrees

    # Format as the standard DMS string: D°M'S.SS"
    return f"{degrees}\u00b0{minutes}'{seconds:.2f}\""


# --- Quick self-test ---
# Verify the function works correctly before processing the full dataset.
test_dd = 103.6922
print(f"Test: {test_dd}° → {decimal_to_dms(test_dd)}")
test_dd_neg = -0.6691
print(f"Test: {test_dd_neg}° → {decimal_to_dms(test_dd_neg)}")

## Step 3: Load the Input CSV

Set the path to your input CSV file. The file must contain at least two columns named `latitude` and `longitude` with decimal degree values.

In [ ]:
# --- USER CONFIGURATION ---
# Update this path to point to your own CSV file.
INPUT_CSV_PATH = "C:/Users/ACER/Downloads/sungaiundan_new.csv"
# --------------------------

# Read the CSV into a DataFrame — pandas' equivalent of opening a spreadsheet in QGIS
location_df = pd.read_csv(INPUT_CSV_PATH)

# Preview the first few rows to confirm the file loaded correctly
# and identify the column names we need (latitude, longitude)
print(f"Loaded {len(location_df)} rows with columns: {list(location_df.columns)}")
location_df.head()

## Step 4: Apply the Conversion

We use `DataFrame.apply()` to run our `decimal_to_dms` function on every row in the `latitude` and `longitude` columns simultaneously — no need to loop manually.

In [ ]:
# Apply the conversion function to every value in the 'latitude' column.
# This creates a new column 'latitude_dms' with the formatted DMS strings.
location_df['latitude_dms'] = location_df['latitude'].apply(decimal_to_dms)

# Same for longitude — results go into a new 'longitude_dms' column.
location_df['longitude_dms'] = location_df['longitude'].apply(decimal_to_dms)

# Preview a few rows to spot-check the results before saving
print("Sample output (first 5 rows):")
location_df[['latitude', 'longitude', 'latitude_dms', 'longitude_dms']].head()

## Step 5: Save the Result

Export the updated DataFrame (with the new DMS columns) to a new CSV file. We use `index=False` to avoid writing the row numbers as an extra column.

In [ ]:
# --- USER CONFIGURATION ---
# Change this to your desired output file path.
OUTPUT_CSV_PATH = "updated_dataset_sungaiundan.csv"
# --------------------------

# Save the enriched DataFrame to disk.
# index=False prevents pandas from writing the integer row index as an extra column.
location_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"✅ Conversion complete!")
print(f"   Rows processed : {len(location_df)}")
print(f"   Output saved to: {OUTPUT_CSV_PATH}")